# Revisión y mejora automática de `agent.md`

Este notebook ejecuta un flujo completo para:
1. Cargar y respaldar `agent.md`.
2. Validar estructura mínima.
3. Detectar mejoras automáticas.
4. Aplicar transformaciones seguras.
5. Mostrar diff unificado.
6. Generar pruebas para futuras ediciones.

## 1) Configurar entorno y rutas del repositorio

Definimos rutas, verificamos existencia de `agent.md` e inicializamos utilidades base.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import json
import difflib

try:
    import yaml
except Exception:
    yaml = None

repo_root = Path.cwd()
agent_path = repo_root / "agent.md"
backup_dir = repo_root / "backups"
backup_dir.mkdir(exist_ok=True)

print(f"Repo: {repo_root}")
print(f"agent.md existe: {agent_path.exists()}")
if not agent_path.exists():
    raise FileNotFoundError(f"No se encontró: {agent_path}")

Repo: /Users/santi/Desktop/proyectoML3
agent.md existe: True


## 2) Cargar y respaldar `agent.md`

Leemos el archivo, mostramos resumen y guardamos backup con timestamp antes de modificar.

In [2]:
original_text = agent_path.read_text(encoding="utf-8")
lines = original_text.splitlines()
word_count = len(re.findall(r"\S+", original_text))

backup_name = f"agent_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
backup_path = backup_dir / backup_name
backup_path.write_text(original_text, encoding="utf-8")

print(f"Caracteres: {len(original_text)}")
print(f"Líneas: {len(lines)}")
print(f"Palabras (aprox): {word_count}")
print(f"Backup: {backup_path}")

Caracteres: 2734
Líneas: 13
Palabras (aprox): 428
Backup: /Users/santi/Desktop/proyectoML3/backups/agent_20260324_202511.md


## 3) Parsear frontmatter YAML y cuerpo Markdown

Separamos frontmatter/cuerpo con regex robusta y normalizamos saltos de línea.

In [ ]:
def normalize_newlines(text: str) -> str:
    return text.replace("\r\n", "\n").replace("\r", "\n")


def split_frontmatter_and_body(text: str):
    text = normalize_newlines(text)
    pattern = r"\A---\n(.*?)\n---\n?(.*)\Z"
    m = re.match(pattern, text, flags=re.DOTALL)
    if m:
        frontmatter_raw, body = m.group(1), m.group(2)
    else:
        frontmatter_raw, body = "", text

    if yaml and frontmatter_raw.strip():
        try:
            fm = yaml.safe_load(frontmatter_raw) or {}
            if not isinstance(fm, dict):
                fm = {}
        except Exception:
            fm = {}
    else:
        fm = {}

    return fm, frontmatter_raw, body.strip() + "\n"

fm_dict, fm_raw, body_md = split_frontmatter_and_body(original_text)
print("Frontmatter detectado:", bool(fm_raw.strip()))
print("Claves frontmatter:", list(fm_dict.keys()))
print("Líneas cuerpo:", len(body_md.splitlines()))

## 4) Validar estructura mínima de `agent.md`

Validamos frontmatter requerido, encabezados esperados y formato básico de listas/encabezados.

In [ ]:
REQUIRED_FRONTMATTER_KEYS = ["name", "description"]
EXPECTED_HEADINGS = [
    "Objetivo",
    "Datos",
    "Metodología",
    "Evaluación",
    "Entregables",
]


def extract_h1_h2_headings(body: str):
    headings = []
    for line in body.splitlines():
        if re.match(r"^#{1,2}\s+", line.strip()):
            headings.append(re.sub(r"^#{1,2}\s+", "", line.strip()))
    return headings


def validate_agent_structure(frontmatter: dict, body: str):
    issues = []
    warnings = []

    if not frontmatter:
        warnings.append("No hay frontmatter YAML.")
    else:
        for key in REQUIRED_FRONTMATTER_KEYS:
            if key not in frontmatter or not str(frontmatter.get(key, "")).strip():
                issues.append(f"Falta campo de frontmatter: {key}")

    headings = extract_h1_h2_headings(body)
    missing = [h for h in EXPECTED_HEADINGS if h.lower() not in {x.lower() for x in headings}]
    if missing:
        warnings.append(f"Secciones recomendadas ausentes: {', '.join(missing)}")

    bad_list_lines = [
        i + 1
        for i, line in enumerate(body.splitlines())
        if re.match(r"^\s*[•]\s+", line)
    ]
    if bad_list_lines:
        warnings.append(f"Viñetas no estándar (usar '-' o enumeración) en líneas: {bad_list_lines[:10]}")

    return issues, warnings, headings

issues, warnings, headings = validate_agent_structure(fm_dict, body_md)
print("Issues:", issues if issues else "Ninguno")
print("Warnings:", warnings if warnings else "Ninguno")
print("Headings detectados:", headings[:10])

## 5) Detectar mejoras automáticas (lint + reglas)

Aplicamos reglas para claridad, deduplicación y consistencia de estilo.

In [ ]:
def dedupe_paragraphs(text: str):
    chunks = [c.strip() for c in re.split(r"\n\s*\n", text) if c.strip()]
    seen = set()
    out = []
    removed = 0
    for ch in chunks:
        norm = re.sub(r"\s+", " ", ch.lower()).strip()
        if norm in seen:
            removed += 1
            continue
        seen.add(norm)
        out.append(ch)
    return "\n\n".join(out) + "\n", removed


def make_structured_body_from_free_text(body: str) -> str:
    body = normalize_newlines(body).strip()
    body, removed = dedupe_paragraphs(body)

    template = f"""# Objetivo
Identificar causas de fuga de empleados y proponer acciones concretas y costo-efectivas para mejorar la retención.

# Datos
- Fuente: `dataSet_RRHH.csv`
- Variables: numéricas y categóricas.
- Requisito: documentar decisiones de limpieza y transformación.

# Metodología
1. Preprocesar variables numéricas y categóricas.
2. Categorizar variables numéricas en niveles (bajo/medio/alto) cuando aporte interpretabilidad.
3. Escalar variables numéricas con `StandardScaler` y `MinMaxScaler` para comparar sensibilidad de algoritmos.
4. Integrar variables transformadas y aplicar one-hot encoding, eliminando redundancias.
5. Evaluar uso de PCA para reducción dimensional e interpretabilidad.
6. Entrenar y comparar clustering con KMeans, K-Prototypes y DBSCAN.

# Evaluación
- Métricas recomendadas: silhouette, inertia (KMeans), y DBCV cuando aplique a clustering por densidad.
- Interpretar cada cluster en lenguaje de negocio: perfil del grupo, posible causa de fuga y riesgo asociado.
- Evitar decisiones basadas únicamente en una métrica; priorizar explicabilidad y accionabilidad.

# Entregables
- Notebook ejecutable (`.ipynb`) con código claro y comentado.
- Visualizaciones consistentes y legibles.
- Resumen para presentación (máximo 10 láminas) con hallazgos y plan de acción.
- Recomendaciones concretas por segmento de empleados.

# Notas
- Se eliminaron {removed} párrafos duplicados durante la normalización automática.
- Referencia visual disponible en `mermaidPizarra.mmd`.
"""
    return template

lint_report = {
    "issues": issues,
    "warnings": warnings,
    "body_length": len(body_md),
}
print(json.dumps(lint_report, indent=2, ensure_ascii=False))

## 6) Aplicar mejoras y regenerar `agent.md`

Reconstruimos frontmatter + cuerpo y escribimos versión mejorada preservando compatibilidad.

In [ ]:
default_frontmatter = {
    "name": "employee-retention-analysis-agent",
    "description": "Guía para analizar causas de fuga de empleados y proponer acciones de retención accionables.",
}

if fm_dict:
    merged_fm = {**default_frontmatter, **fm_dict}
else:
    merged_fm = default_frontmatter

improved_body = make_structured_body_from_free_text(body_md)

if yaml:
    fm_text = yaml.safe_dump(merged_fm, sort_keys=False, allow_unicode=True).strip()
else:
    # Fallback YAML simple.
    fm_text = "\n".join([f"{k}: {json.dumps(v, ensure_ascii=False)}" for k, v in merged_fm.items()])

improved_text = f"---\n{fm_text}\n---\n\n{improved_body}".rstrip() + "\n"
agent_path.write_text(improved_text, encoding="utf-8")

print("agent.md actualizado.")
print("Nuevas claves frontmatter:", list(merged_fm.keys()))
print("Líneas finales:", len(improved_text.splitlines()))

## 7) Comparar cambios con diff unificado

Mostramos diferencias para revisión rápida antes de commit.

In [ ]:
updated_text = agent_path.read_text(encoding="utf-8")
diff = difflib.unified_diff(
    original_text.splitlines(),
    updated_text.splitlines(),
    fromfile="agent.md (original)",
    tofile="agent.md (mejorado)",
    lineterm="",
)

diff_text = "\n".join(diff)
print(diff_text[:8000] if diff_text else "Sin cambios detectados")

## 8) Agregar pruebas de validación para futuras ediciones

Generamos pruebas unitarias (casos positivos y negativos) y ejecutamos validación básica.

In [ ]:
test_code = '''import re
from pathlib import Path


def has_frontmatter(text: str) -> bool:
    return bool(re.match(r"\\A---\\n.*?\\n---\\n", text, flags=re.DOTALL))


def test_agent_md_exists():
    assert Path("agent.md").exists(), "agent.md no existe"


def test_agent_md_has_frontmatter_and_sections():
    text = Path("agent.md").read_text(encoding="utf-8")
    assert has_frontmatter(text), "Falta frontmatter YAML"
    for section in ["# Objetivo", "# Datos", "# Metodología", "# Evaluación", "# Entregables"]:
        assert section in text, f"Falta sección: {section}"


def test_negative_missing_section_fails():
    broken = "---\\nname: x\\ndescription: y\\n---\\n\\n# Objetivo\\n"
    assert "# Entregables" not in broken
'''

tests_path = repo_root / "test_agent_md.py"
tests_path.write_text(test_code, encoding="utf-8")
print(f"Test creado: {tests_path}")

# Ejecución rápida de prueba local
import subprocess
res = subprocess.run([
    str(repo_root / ".venv" / "bin" / "python"), "-m", "pytest", "-q", str(tests_path)
], capture_output=True, text=True)
print(res.stdout)
print(res.stderr)
print("exit_code:", res.returncode)